# Figure 4 — three-colour tracking (b, c) and single-molecule FRET (e–h)

**Makes up:**
- **b**: one exemplar track of each protein (ICAM-1, CD58, UCHT1-Fab), 500 nm scale bar;
- **c**: distribution of the diffusion coefficient D per protein (median ± MAD, N);
- **e**: one raw frame of the FRET field of view (ZWO), 20 µm scale bar, zoom box;
- **f**: the zoom, Bayer key, the three example puncta circled and numbered;
- **g**: the three puncta's A_R / A_G traces with change points and segment means;
- **h**: dwell-time histograms of the low- and high-FRET states with exponential fits
  (k_Low→High, k_High→Low).
- **SI Figs. S84–S88** (section 8): E_App states, spectral-shift resolution against photoelectrons, per-frame
  pixel QE states, resolution against frames per state, and six example fits with their Bayer keys.

Panels a and d are cartoons.

**Replaces** `tracking/20260331_Dan_RawData_Analysis` + `tracking/20260331_Dan_Track_Analysis` (b, c) and
`fret/DNA_HJ_Analysis_ZWO` → `fret/DNA_HJ_PostHoc_Changepoints_AllFiles` → `fret/FRET_FigureMaking_btc29_zwo`
(e–h). Shared code in `_tracking_fret.py`.

**Changes from Brendan's FRET notebook (h):**
- The states come from the **two-component Gaussian mixture on the segment FRET efficiencies** (his cell 48), not
  from the sign of the colour step (`DeltaR`). In his notebook that mixture was overwritten by the later
  per-pixel mixtures on A_R and A_G, and `exposure_time` by a 0.1 s Ximea leftover (the data is 10 ms).
- Consecutive segments in the same state are one dwell; a dwell runs from its first frame to the first frame of
  the next; the first and last dwell of each trace (cut by the observation) are dropped.
- All 25 positions and every punctum are used (his loop skipped the last file and the last punctum of each).
- The binned exponential fit uses Poisson weights (√counts; his used 1/counts²), and the maximum-likelihood rate
  of the truncated exponential is printed beside it.
- `N_molecules` is the number of puncta contributing dwells (his printed the number of change points).

**Data:** BioStudies `S-BIAD3210`. Tracking: `2026_Multicolour_Paper/Ximea/LipidBilayers/AF488ICAM1_AF555CD58_JF646UCHT1/`, 31 fields of view (1 nM, 250 pM and 500 pM total protein; wells and fields as sub-folders), one TIFF (~0.97 GB) and one MicroManager metadata file each. FRET: `2026_Multicolour_Paper/ZWO/20260402_HJs/50mM_MgCl_20pMHJ_515LP650200BP_100perc515_2/`, 25 positions (~2.3 GB each). The deposition holds raw data only, so this notebook fits it first.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits the two tracking fields of view that hold the exemplar tracks of b and the pictured FRET position (`Pos-0-0`): ~4.3 GB. Every panel is made, from fewer molecules: c has ~1/15 of the tracks and h the dwells of one position in 25. `FAST_MODE = False` uses all of it (~90 GB, many hours of fitting; run in `tmux`/`screen`).

**Run order:** the fits (section 2) take hours. The first run fits both datasets (`REFIT_* = "auto"`: fit only when a TIFF has no fit made after the x/y fix of 2026-10-28; later runs
skip straight to the analysis). `REFIT_* = True` forces a refit, moving existing pre-fix fits to a sibling
`old_fits/` folder. Change points are recomputed when missing or older than the fit. Fits are written beside the TIFFs in `DATA_ROOT`; tables and figures go to `OUT_ROOT`.

**Plotting only:** each analysis step saves its result (tracks, change points, FRET segments), and every panel's cell reads it back, so once they exist run the setup and download cells (the download skips complete files) and then any panel.

---
## 0 — Setup

In [ ]:
import sys
import types
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from scipy.stats import median_abs_deviation

sys.path.insert(0, str(Path.cwd().resolve()))
import _biostudies_download as bd
import _tracking_fret as tf

from pyS3M import IOFunctions, PlottingBase, SpectralFunctions, SR_Functions, SM_extractionfunctions, sCMOSFunctions
from pyS3M.diffusion import MSD

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

FAST_MODE = True   # True: the exemplars' 2 tracking fields + FRET Pos-0-0 (~4.3 GB). False: all of it (~90 GB).

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB = REPO_ROOT / "Camera_Calibrations"
OUT_ROOT  = Path("outputs") / "Figure_4_Tracking_FRET"   # git-ignored (bare outputs/ rule)
RUN = "_fast" if FAST_MODE else ""
FIG_DIR = OUT_ROOT / f"figures{RUN}"
CP_MIN_SIZE = 10   # minimum change-point segment (frames): 0.1 s
CP_DIR = OUT_ROOT / f"fret_changepoints_min{CP_MIN_SIZE}{RUN}"
# saved by the analysis cells, read back by the panels
TRACK_SM_H5 = OUT_ROOT / f"tracking_single_molecules{RUN}.h5"
TRACK_SF_H5 = OUT_ROOT / f"tracking_single_frames{RUN}.h5"
FRET_SEG_H5 = OUT_ROOT / f"fret_segments_min{CP_MIN_SIZE}{RUN}.h5"
# Point DATA_ROOT at a drive with room for the download (one folder per FAST_MODE setting: the fits
# read every TIFF in it).
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
CP_DIR.mkdir(parents=True, exist_ok=True)

# Tracking: Ximea, 69 nm, 50 ms frames. FRET: ZWO (RGGB), 71.5 nm, 10 ms frames.
TRACK_PIXEL_NM, TRACK_DT = 69.0, 0.05
FRET_PIXEL_NM, FRET_DT = 71.5, 0.01

REFIT_TRACKING = "auto"   # "auto": fit if any TIFF lacks a current fit; True: move pre-fix fits to old_fits/ and refit
REFIT_FRET = "auto"       # (each fit takes hours)


def needs_fit(flag, root):
    return flag is True or (flag == "auto" and not all(tf.fit_is_current(f) for _, f in tf.tiff_fit_pairs(root)))


def smoothing():
    return types.SimpleNamespace(args={"sigma": 1.5}, extent=1.5, data_arg="image",
                                 smoothing_function=sCMOS.gaussian_filter_stack)


def calibration(camera_dir):
    return {k: IO.read_tiff(CALIB / camera_dir / f"{f}.tif") for k, f in
            [("gain_map", "gain"), ("offset_map", "offset"), ("variance", "variance"),
             ("read_noise", "readnoise"), ("rqe", "rqe")]}


# FRET camera and calibration (used by the fit and by e/f)
MOSAIC_ZWO = np.array([["R", "G"], ["G", "B"]])
SupRes_Z = SR_Functions.SuperRes_Functions(camera="zwo", mosaic_unit=MOSAIC_ZWO)
cal_zwo = calibration("ZWO_Camera")


def savefig(fig, name):
    fig.savefig(FIG_DIR / name, dpi=600, format="svg", bbox_inches="tight")


print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 1 — Download

Each tracking field of view into its own sub-folder of `DATA_ROOT/Tracking` (as deposited), the FRET positions into `DATA_ROOT/FRET`, each TIFF with its metadata (the camera ROI the fit needs). Re-running skips complete files.

In [ ]:
track_fovs = tf.TRACK_FAST_FOVS if FAST_MODE else tf.TRACK_FOVS
track_tifs = []
for fov in track_fovs:
    track_tifs += bd.download_acquisition(f"{tf.TRACK_REMOTE}/{fov}", DATA_ROOT / "Tracking" / fov)["images"]
TRACK_ROOT = track_tifs[0].parents[len(Path(track_fovs[0]).parts)]   # holds the fields' folders

fret_files = bd.download_acquisition(tf.FRET_REMOTE, DATA_ROOT / "FRET",
                                     stem=tf.FRET_FAST_STEM if FAST_MODE else None)
FRET_ROOT = fret_files["images"][0].parent
# each position's change-point file (section 4 makes them)
cp_files = [(tif, CP_DIR / (fit.stem + "_colourchangepoint.h5")) for tif, fit in tf.tiff_fit_pairs(FRET_ROOT)]
print(f"tracking: {len(track_tifs)} field(s) of view under {TRACK_ROOT}")
print(f"FRET: {len(fret_files['images'])} position(s) in {FRET_ROOT}")

---
## 2 — Fits

Tracking: `fit_tracking_data` on every TIFF under `TRACK_ROOT` (as `20260331_Dan_RawData_Analysis`). FRET:
`fit_FRET_data` on the 25 positions (as `DNA_HJ_Analysis_ZWO`). Each writes `<tiff>.h5` beside the TIFF.

In [ ]:
track_pairs = tf.tiff_fit_pairs(TRACK_ROOT)
fret_pairs = tf.tiff_fit_pairs(FRET_ROOT)
for label, pairs in [("tracking", track_pairs), ("FRET", fret_pairs)]:
    n_ok = sum(tf.fit_is_current(f) for _, f in pairs)
    print(f"{label}: {len(pairs)} TIFFs, {n_ok} with a fit made after {tf.FIT_FIX:%Y-%m-%d}")
print("pre-fix fits that REFIT_* would move:")
for root in (TRACK_ROOT, FRET_ROOT):
    for src, dst in tf.move_old_fits(root, dry_run=True):
        print(f"  {src.relative_to(root.parent)} -> {dst.relative_to(root.parent)}")

In [ ]:
# Tracking fit (Ximea defaults)
if needs_fit(REFIT_TRACKING, TRACK_ROOT):
    tf.move_old_fits(TRACK_ROOT)
    SupRes_X = SR_Functions.SuperRes_Functions(camera="ximea")
    cal = calibration("Ximea_Camera")
    SupRes_X.fit_tracking_data(
        str(TRACK_ROOT), smoothing(), cal["gain_map"], cal["offset_map"], cal["rqe"], cal["read_noise"],
        variance=cal["variance"], pfa=1e-3, ROI_size=20, peak_wavelength=0.6, fraction_true=0.0,
        sigma=1.5, NA=1.49, pixel_size=TRACK_PIXEL_NM / 1000, image_type=".tif")
missing = [t.name for t, f in tf.tiff_fit_pairs(TRACK_ROOT) if not tf.fit_is_current(f)]
assert not missing, f"tracking: {len(missing)} TIFFs without a current fit; set REFIT_TRACKING = True"

In [ ]:
# FRET fit (ZWO, RGGB)
if needs_fit(REFIT_FRET, FRET_ROOT):
    tf.move_old_fits(FRET_ROOT)
    SupRes_Z.fit_FRET_data(
        str(FRET_ROOT), smoothing(), gain_map=cal_zwo["gain_map"], offset_map=cal_zwo["offset_map"],
        variance=cal_zwo["variance"], rqe=cal_zwo["rqe"], read_noise=cal_zwo["read_noise"], pfa=1e-3,
        ROI_size=14, peak_wavelength=0.6, sigma=1.5, fraction_true=0.0, NA=1.49,
        pixel_size=FRET_PIXEL_NM / 1000, image_type=".tif", n_frames_sum=50)
missing = [t.name for t, f in tf.tiff_fit_pairs(FRET_ROOT) if not tf.fit_is_current(f)]
assert not missing, f"FRET: {len(missing)} TIFFs without a current fit; set REFIT_FRET = True"

---
## 3 — Tracking (b, c)

Spectral LAP linking per field of view (Jaqaman et al. 2008 with a colour term), parameters as
`20260331_Dan_Track_Analysis`; then KMeans into the three proteins, and D per track by the OLSF MSD fit.

In [ ]:
LINK = dict(max_distance=10.0, max_dark_time=5, w_spatial=1.0, w_spectral=1.0, spectral_tol=0.25,
            chi_val=2.0, max_localisation_error=1.5, max_colour_error=1.0, min_photons=100, max_photons=1e6,
            min_frames=10, D_prior=0.1 * 1e6 / TRACK_PIXEL_NM**2, dt=TRACK_DT, sigma_loc=50.0 / TRACK_PIXEL_NM,
            alpha=4.0, remove_static=True, static_eps=None, static_min_samples=20, verbose=False,
            # PSF sigma cuts of the paper's run (75-160 nm, sigma error < 40 nm). The default upper cut is now
            # 250 nm; at 1 nM the broader spots let the linker hop between molecules (D ~1 instead of ~0.1).
            min_sigma=75.0 / TRACK_PIXEL_NM, max_sigma=160.0 / TRACK_PIXEL_NM, max_sigma_error=40.0 / TRACK_PIXEL_NM)

SM_E = SM_extractionfunctions.extract_SMs()
all_sm, all_sf, offset = [], [], 0
for tif, fit in tf.tiff_fit_pairs(TRACK_ROOT):
    sm, sf = SM_E.extract_single_molecules_spectral_lap(pd.read_hdf(fit), **LINK)
    print(f"{fit.relative_to(TRACK_ROOT)}: {len(sm)} molecules")
    if not len(sm):
        continue
    for d in (sm, sf):
        d["molecular_index"] += offset
        d["fov_name"] = str(fit)
    offset = int(sm["molecular_index"].max()) + 1
    all_sm.append(sm)
    all_sf.append(sf)
sm_db = pd.concat(all_sm, ignore_index=True)
sf_db = pd.concat(all_sf, ignore_index=True)

sm_db["colour_cluster"] = tf.colour_clusters(sm_db)
sf_db["colour_cluster"] = sf_db["molecular_index"].map(sm_db.set_index("molecular_index")["colour_cluster"])

D, s2 = tf.diffusion_per_track(sm_db, sf_db, MSD(), TRACK_PIXEL_NM, TRACK_DT, min_frames=5)
sm_db["D_um2_s"], sm_db["sigma2_nm2"] = D, s2
mobile = sm_db["D_um2_s"].notna() & sm_db["D_um2_s"].between(1e-4, 1000, inclusive="neither")
print(f"mobile: {mobile.sum()} / {len(sm_db)} molecules")
sm_db = sm_db[mobile].reset_index(drop=True)
sf_db = sf_db[sf_db["molecular_index"].isin(sm_db["molecular_index"])].copy()

sm_db.to_hdf(TRACK_SM_H5, key="data", format="table", mode="w")
sf_db.to_hdf(TRACK_SF_H5, key="data", format="table", mode="w")
sm_db.groupby("colour_cluster")["D_um2_s"].describe()

### b — exemplar tracks

One track per protein (`tf.EXEMPLARS`: field of view, first and last frame, position), each centred on its own median position. They are the molecules and frames of the composite Figure 4b; D is that of the whole track.

In [ ]:
sm_db, sf_db = pd.read_hdf(TRACK_SM_H5), pd.read_hdf(TRACK_SF_H5)   # section 3
um = TRACK_PIXEL_NM / 1000

tracks = {k: g.sort_values("frame") for k, g in sf_db.groupby("molecular_index")}
fig, (ax,) = tf.mm_stack(*tf.PANEL_MM["b"], margins_mm=(1, 1, 1, 1))   # axes as placed in the paper
for k in range(3):
    fov, first_frame, last_frame, xy = tf.EXEMPLARS[k]
    mid = tf.find_exemplar(sm_db, sf_db, TRACK_ROOT, fov, first_frame, xy)
    tr = tracks[mid][tracks[mid]["frame"].between(first_frame, last_frame)]
    xx = um * (tr["xc"] - tr["xc"].median())
    yy = um * (tr["yc"] - tr["yc"].median())
    ax.plot(xx, yy, "-o", ms=2.5, lw=0.8, color=tf.CLUSTER_COLOURS[k], alpha=tf.CLUSTER_ALPHAS[k])   # as c
    print(f"{tf.CLUSTER_NAMES[k]}: molecule {mid} ({fov}, frames {tr['frame'].min()}-{tr['frame'].max()} of "
          f"{tracks[mid]['frame'].min()}-{tracks[mid]['frame'].max()}), {len(tr)} points, track D = {sm_db.set_index('molecular_index').loc[mid, 'D_um2_s']:.3f} µm² s⁻¹, "
          f"assigned to {tf.CLUSTER_NAMES[sm_db.set_index('molecular_index').loc[mid, 'colour_cluster']]}")
ax.set_aspect("equal", adjustable="datalim")   # keeps the axes size
ax.set_xticks([]), ax.set_yticks([])
ax.add_artist(AnchoredSizeBar(ax.transData, 0.5, "500 nm", "lower center", pad=0.1, color="black",
                              frameon=False, fontproperties={"size": 6}))
savefig(fig, "Fig4b_exemplar_tracks.svg")
plt.show()

### c — D per protein (median ± MAD)

In [ ]:
sm_db = pd.read_hdf(TRACK_SM_H5)   # section 3
logD = np.log10(sm_db["D_um2_s"])
bins = 10 ** np.histogram_bin_edges(logD, "fd")
fig, (ax,) = tf.mm_stack(*tf.PANEL_MM["c"], margins_mm=(12, 9, 1, 1))
for k, alpha in enumerate(tf.CLUSTER_ALPHAS):
    v = sm_db.loc[sm_db["colour_cluster"] == k, "D_um2_s"]
    ax.hist(v, bins=bins, color=tf.CLUSTER_COLOURS[k], alpha=alpha, edgecolor="black", linewidth=0.3,
            label=f"{tf.CLUSTER_NAMES[k]}, D = {np.median(v):.2f} ± {median_abs_deviation(v):.2f} (N = {len(v)})")
ax.set_xscale("log")
ax.set_xlabel(r"D / µm$^2$ s$^{-1}$")
ax.set_ylabel("frequency")
ax.legend(fontsize=6)
ax.grid(True, color="gray", lw=0.5, alpha=0.5, ls="--")
savefig(fig, "Fig4c_D_distribution.svg")
plt.show()

---
## 4 — FRET change points

Per position: drop never-fitted rows, then joint (A_R, A_G) change points per punctum (PELT, l2 cost, minimum
segment `CP_MIN_SIZE` = 10 frames = 0.1 s, BIC penalty), as `DNA_HJ_PostHoc_Changepoints_AllFiles` (which used
25 frames; with 0.1 s, more of the short dwells are resolved). Written to
`CP_DIR/<position>_colourchangepoint.h5` (data, changepoints, segments).

In [ ]:
for tif, out in cp_files:
    fit = tif.with_name(tif.name.split(".")[0] + ".h5")
    if not (out.exists() and out.stat().st_mtime > fit.stat().st_mtime):
        df, n_guess = tf.filter_initial_guess(pd.read_hdf(fit))
        cps = tf.change_points(df, min_size=CP_MIN_SIZE)
        tf.save_changepoints(out, df, cps)
        print(f"{fit.stem}: {df['puncta_id'].nunique()} puncta, {len(cps)} with change points "
              f"({n_guess} initial-guess rows dropped)")

---
## 5 — FRET efficiency and the two states

Apparent E: projection of each segment's (A_B, A_G, A_R) onto the line from the Cy3 to the Cy5 pixel fractions
(ZWO QE, 514 nm dichroic, 650/200 band-pass, 515 long-pass); then gamma-corrected for detection efficiency and
quantum yield (Cy3 0.16, Cy5 0.29). A two-component Gaussian mixture on the segment E values assigns each
segment to the low- or high-FRET state.

In [ ]:
FILTERS = ["semrock-di03-r514-t1-25x36", "semrock-ff01-650-200", "semrock-ff01-515-lp"]
S_Fz = SpectralFunctions.Spectral_Funcs(camera="zwo")
f_cy3, f_cy5, gamma = tf.cy3_cy5_fingerprints(S_Fz, FILTERS)
print(f"Cy3 (B, G, R) {np.round(f_cy3, 3)}; Cy5 {np.round(f_cy5, 3)}; gamma {gamma:.3f}")

segs = []
for i, (tif, cpf) in enumerate(cp_files):
    _, _, s = tf.load_changepoints(cpf)
    s["position"] = i
    s["puncta_id"] = s["puncta_id"].astype(str).radd(f"{i}_")   # unique across positions
    segs.append(s)
seg = pd.concat(segs, ignore_index=True)
E_app, _ = tf.apparent_E(seg[["A_B_mean", "A_G_mean", "A_R_mean"]].to_numpy(float), f_cy3, f_cy5)
seg["E"] = tf.gamma_correct(E_app, gamma)

gmm, HIGH = tf.two_state_gmm(seg["E"])
inside = seg["E"].between(0, 1)
seg["state"] = -1
seg.loc[inside, "state"] = gmm.predict(seg.loc[inside, ["E"]].to_numpy())
means = gmm.means_.ravel()
seg.to_hdf(FRET_SEG_H5, key="segments", format="table", mode="w")
pd.Series({"high_label": HIGH, "E_low": means[1 - HIGH], "E_high": means[HIGH]}).to_hdf(FRET_SEG_H5, key="states")
print(f"{len(seg):,} segments ({(~inside).sum()} with E outside [0, 1] left unassigned); "
      f"low-FRET E = {means[1 - HIGH]:.2f}, high-FRET E = {means[HIGH]:.2f}")

xs = np.linspace(0, 1, 500)
fig, ax = plotter.one_column_plot(height=2, width=3)
ax.hist(seg.loc[inside, "E"], bins=51, range=(0, 1), density=True, color="0.7")
for k, colour in [(1 - HIGH, "#000080"), (HIGH, "darkorange")]:
    sd = np.sqrt(gmm.covariances_.ravel()[k])
    ax.plot(xs, gmm.weights_[k] * np.exp(-0.5 * ((xs - means[k]) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)),
            color=colour, lw=1.5)
    ax.axvline(means[k], color=colour, ls="--", lw=1)
ax.set(xlabel=r"$E_{FRET}$", ylabel="probability density")
savefig(fig, "Fig4_FRET_E_states.svg")
plt.show()

---
## 6 — e, f: field of view and example puncta

`EXAMPLE_XY` (camera px, `xc`, `yc`): the three circled puncta of the paper's zoom (`ZoomedFOV_SelectPuncta.svg`,
read from its axes; re-picked by position, since a refit renumbers `puncta_id`); the nearest punctum with change
points in position 0 is used. Puncta 1 and 3 are the paper's (found within 1 px). The paper's punctum 2
(503.3, 123.3) is not detected in the refit, and the nearest one (506.3, 81.7) stops switching after 3.3 s, so 2 is
the punctum of the zoom box that keeps switching longest: (402.0, 186.1), 11 change points over 9.7 s, no
stretch longer than 2.8 s without one.

e and f are in photoelectrons on one scale (0.5-99.9th percentile of the frame); the colour bar beside the
zoom (f) applies to both.

In [ ]:
FRAME = 200
BOX = dict(x0=300, y0=0, size=225)                      # zoom box (px)
EXAMPLE_XY = [(459.7, 82.3), (402.0, 186.1), (330.6, 147.6)]   # 1, 3 as the paper; 2 re-picked (see above)

tif0, cpf0 = cp_files[0]
data0, cps0, seg0 = tf.load_changepoints(cpf0)
centres = data0.groupby("puncta_id")[["xc", "yc"]].mean()
centres = centres[centres.index.isin(seg0["puncta_id"].unique())]
examples = []
for x, y in EXAMPLE_XY:
    d = np.hypot(centres["xc"] - x, centres["yc"] - y)
    examples.append(d.idxmin())
    print(f"({x}, {y}) -> punctum {d.idxmin()} at ({centres.loc[d.idxmin(), 'xc']:.1f}, "
          f"{centres.loc[d.idxmin(), 'yc']:.1f}), {d.min():.1f} px away")
    cx, cy = centres.loc[d.idxmin(), ["xc", "yc"]]
    assert BOX["x0"] <= cx < BOX["x0"] + BOX["size"] and BOX["y0"] <= cy < BOX["y0"] + BOX["size"], "outside the zoom"

raw = IO.read_tiff(str(tif0), frame=FRAME)
meta = sorted(FRET_ROOT.glob(tif0.name.split(".")[0] + "_metadata.txt"))[0]
roi_x, roi_y, _, _ = IO.metadata_reader_imageJ(str(meta))
# ADU -> photoelectrons with the calibration maps cropped to this position's camera ROI
roi_x, roi_y = int(roi_x), int(roi_y)
roi_maps = {k: cal_zwo[k][roi_y:roi_y + raw.shape[0], roi_x:roi_x + raw.shape[1]] for k in ("gain_map", "offset_map", "rqe")}
pe = IO.convert_to_photoelectrons(raw.astype(np.float32), **roi_maps)
VMIN, VMAX = np.percentile(pe, [0.5, 99.9])           # one photoelectron scale for e and f
full_masks = SupRes_Z.mask.get_masks(*cal_zwo["gain_map"].shape)

# e: full field of view
fig, (ax,) = tf.mm_stack(*tf.PANEL_MM["e"], margins_mm=(1, 1, 1, 1))
plotter.image_plot(ax, pe, cmap="gray", origin="upper", vmin=VMIN, vmax=VMAX,
                   scalebar=True, pixelsize=FRET_PIXEL_NM, scalebarsize=20000, scalebarlabel="20 μm")
ax.add_patch(plt.Rectangle((BOX["x0"] - 0.5, BOX["y0"] - 0.5), BOX["size"], BOX["size"], fill=False, ec="white", lw=0.5))
savefig(fig, "Fig4e_FullFOV.svg")
plt.show()

# f: zoom, with the Bayer key of the camera pixels under it
x0, y0, n = BOX["x0"], BOX["y0"], BOX["size"]
crop = pe[y0:y0 + n, x0:x0 + n]
fig, (ax,) = tf.mm_stack(*tf.PANEL_MM["f"], margins_mm=(1, 1, 8, 1))
plotter.image_plot(ax, crop, cmap="gray", origin="upper", vmin=VMIN, vmax=VMAX,      # same scale as e
                   scalebar=True, pixelsize=FRET_PIXEL_NM, scalebarsize=5000, scalebarlabel="5 μm")
tf.side_colorbar(fig, ax, VMIN, VMAX, "photoelectrons")             # applies to e and f
PlottingBase.add_bayer_key(ax, full_masks, row_min=int(roi_y) + y0, col_min=int(roi_x) + x0)
for i, pid in enumerate(examples, start=1):
    cx, cy = centres.loc[pid, "xc"] - x0, centres.loc[pid, "yc"] - y0
    ax.add_patch(plt.Circle((cx, cy), 10, fill=False, ec="white", lw=1))
    ax.text(cx - 14, cy - 12, str(i), color="white", fontsize=7, ha="right")
savefig(fig, "Fig4f_Zoom_SelectPuncta.svg")
plt.show()

### g — traces

A_R (red) and A_G (green) per frame; segment means between change points (dashed vertical lines); the last
segment, cut by the end of the trace, dashed.

In [ ]:
G_W, G_H, G_GAP = tf.PANEL_MM["g"]
fig, axs = tf.mm_stack(G_W, G_H, n=3, gap_mm=G_GAP, margins_mm=(12, 9, 1, 1))
for i, (ax, pid) in enumerate(zip(np.atleast_1d(axs), examples), start=1):
    sub = data0[data0["puncta_id"] == pid].sort_values("frame")
    s = seg0[seg0["puncta_id"] == pid].sort_values("segment_number")
    c = cps0[cps0["puncta_id"] == pid]
    t = sub["frame"].to_numpy() * FRET_DT
    ax.scatter(t, sub["A_R"], s=1, c="red", linewidths=0, alpha=0.75, rasterized=True)
    ax.scatter(t, sub["A_G"], s=1, c="green", linewidths=0, alpha=0.75, rasterized=True)
    for j, (_, row) in enumerate(s.iterrows()):
        ls = "--" if j == len(s) - 1 else "-"
        for col, colour in [("A_R_mean", "red"), ("A_G_mean", "green")]:
            ax.hlines(row[col], row["frame_start"] * FRET_DT, row["frame_end"] * FRET_DT, colors=colour, lw=1, ls=ls)
    for f in c["frame"]:
        ax.axvline(f * FRET_DT, ls="--", color="black", lw=0.8)
    ax.set_xlim(0, s["frame_end"].max() * FRET_DT)
    ax.set_ylim(0.2, 0.8)
    ax.text(0.005, 0.95, str(i), transform=ax.transAxes, fontsize=7, va="top")
    ax.set_xlabel("time/s" if i == len(examples) else "")
    ax.set_ylabel("relative pixel QE" if i == 2 else "")
savefig(fig, "Fig4g_changepoint_traces.svg")
plt.show()

---
## 7 — h: dwell times and rates

Dwells of each mixture state (merged runs, interior only; section 5). The time spent in the low-FRET state gives
k_Low→High, in the high-FRET state k_High→Low. Dwells shorter than one minimum segment (`T_MIN`) cannot occur.

Two fits per state: a single exponential (the binned fit, solid, and its maximum-likelihood rate) and a two-exponential maximum-likelihood fit (dashed: fraction `w` exiting at k_fast, the rest at k_slow), with AIC for it and for one exponential on the same dwells (lower is better). The two-exponential fit starts one frame above `T_MIN` (`T_FIT`): change-point segments pile up at exactly the minimum length (14% of the dwells at 0.25 s), and that spike would be fitted as a fast component.

In [ ]:
seg = pd.read_hdf(FRET_SEG_H5, "segments")   # section 5
HIGH = int(pd.read_hdf(FRET_SEG_H5, "states")["high_label"])
dw = tf.state_dwells(seg[seg["state"] >= 0], HIGH, FRET_DT)
T_MIN = CP_MIN_SIZE * FRET_DT
rates = {s: tf.exp_rate(dw.loc[dw["state"] == s, "dwell_s"], T_MIN) for s in ("low", "high")}
n_mol = dw["puncta_id"].nunique()
for s, label in [("low", "k_Low→High"), ("high", "k_High→Low")]:
    r = rates[s]
    print(f"{label}: binned fit {r['k_fit']:.2f} ± {r['k_fit_err']:.2f} s⁻¹, "
          f"MLE {r['k_mle']:.2f} ± {r['k_mle_err']:.2f} s⁻¹ ({r['n']:,} dwells)")
print(f"N_molecules = {n_mol:,} puncta with at least one complete dwell")
T_FIT = (CP_MIN_SIZE + 1) * FRET_DT                    # two-exponential fit: above the minimum-length spike
bi = {s: tf.biexp_rate(dw.loc[dw["state"] == s, "dwell_s"], T_FIT) for s in ("low", "high")}
for s, label in [("low", "k_Low→High"), ("high", "k_High→Low")]:
    b = bi[s]
    n_spike = int(np.isclose(dw.loc[dw["state"] == s, "dwell_s"], T_MIN).sum())
    print(f"{label} (dwells ≥ {T_FIT:.2f} s: {b['n']:,}; {n_spike:,} at exactly {T_MIN:.2f} s left out): "
          f"one-exponential MLE {b['k_mono']:.2f} s⁻¹; two-exponential MLE: k_fast {b['k_fast']:.2f} ± {b['k_fast_err']:.2f} s⁻¹ "
          f"({b['w_fast']:.0%} ± {b['w_fast_err']:.0%} of dwells), k_slow {b['k_slow']:.2f} ± {b['k_slow_err']:.2f} s⁻¹; "
          f"AIC two {b['aic_bi']:.0f} vs one {b['aic_mono']:.0f}" + ("" if b["resolved"] else " (not resolved: one exponential)"))

H_W, H_H, H_GAP = tf.PANEL_MM["h"]
fig, axs = tf.mm_stack(H_W, H_H, n=2, gap_mm=H_GAP, margins_mm=(12, 9, 1, 1))
for ax, s, colour, label in [(axs[0], "low", "#000080", r"$k_{\rm Low\to High}$"),
                             (axs[1], "high", "darkorange", r"$k_{\rm High\to Low}$")]:
    r = rates[s]
    centres_ = 0.5 * (r["edges"][:-1] + r["edges"][1:])
    ax.stairs(r["counts"] / 1e3, r["edges"], fill=True, color=colour, alpha=0.5)
    tt = np.linspace(T_MIN, 3, 300)
    ax.plot(tt, r["a"] * np.exp(-r["k_fit"] * tt) / 1e3, color=colour, lw=2)
    b, width = bi[s], np.diff(r["edges"]).mean()          # two-exponential: expected counts per bin
    tb = tt[tt >= T_FIT]
    pdf = b["w_fast"] * b["k_fast"] * np.exp(-b["k_fast"] * (tb - T_FIT)) \
        + (1 - b["w_fast"]) * b["k_slow"] * np.exp(-b["k_slow"] * (tb - T_FIT))
    ax.plot(tb, b["n"] * width * pdf / 1e3, color=colour, lw=1, ls="--")
    ax.text(0.95, 0.9, f"{label} = {r['k_fit']:.2f} ± {r['k_fit_err']:.2f} s$^{{-1}}$", color=colour,
            transform=ax.transAxes, ha="right", va="top", fontsize=7)
    ax.set_xlim(T_MIN, 3)
    ax.set_ylabel(r"# of events/10$^3$")
    ax.grid(True, color="gray", lw=0.5, alpha=0.5, ls="--")
axs[0].set_xlabel("")
axs[0].tick_params(labelbottom=False)   # the gap has no room for them
axs[1].set_xlabel("dwell time/s")
axs[1].text(0.95, 0.1, rf"$N_{{\rm molecules}}$={n_mol:,}", transform=axs[1].transAxes, ha="right", fontsize=6)
savefig(fig, "Fig4h_dwell_times.svg")
plt.show()

---
## 8 — SI figures S84–S88

From the saved change points and segments (sections 4, 5) and the Pos-0-0 fit; each figure is saved under its file name in the SI.

- **S84** (`fig:SI_EFRET`): segment E_App with the two-state mixture of section 5.
- **S86** (`fig:SI_FitPixelQE_Gaussians`): per-frame relative QE of pixel 2 (A_G) and pixel 3 (A_R) of every punctum with a change point, each fitted with two Gaussians. The low-FRET state has the higher A_G and the lower A_R.
- **S85** (`fig:SI_EstimatedSpectralShiftResolution`): the resolvable shift, 2 σ_colour, predicted by Figure 2d's colour-precision fit (Ximea TetraSpeck beads, `tf.BEAD_COLOUR_FIT`) against detected photoelectrons per frame, for the FRET puncta's mean relative pixel QEs; points: 2 × the mean width of S86's two Gaussians, at the puncta's mean photoelectrons per frame.
- **S87** (`fig:SI_deltaPixel_vs_Frames`): the change of relative pixel QE resolvable from N frames in each state, 2 (σ_low + σ_high)/√N (S86 widths); dashed: the observed differences of the state means; black: the smallest N that resolves both; blue/orange: median frames per segment of the low/high state.
- **S88** (`fig:Example_FRET_Fit`): six single-frame fits of Pos-0-0 picked at random among the puncta of the dwell analysis (quality cut as the paper's), cut and weighted as `fit_FRET_data` does; raw and fit images carry the 2 × 2 Bayer key of the sensor pixels under their top-left corner.

In [ ]:
# per-frame relative pixel QEs of the puncta with change points (S85-S87)
seg = pd.read_hdf(FRET_SEG_H5, "segments")   # section 5
HIGH = int(pd.read_hdf(FRET_SEG_H5, "states")["high_label"])
analysed = set(seg["puncta_id"])
frames = []
for i, (tif, cpf) in enumerate(cp_files):
    d = pd.read_hdf(cpf, key="data")[["puncta_id", "A_B", "A_G", "A_R", "photoelectrons"]]
    frames.append(d[d["puncta_id"].astype(str).radd(f"{i}_").isin(analysed)])
frames = pd.concat(frames, ignore_index=True)

PX = {col: tf.pixel_qe_states(frames[col])[1] for col in ("A_G", "A_R")}   # components in increasing mean
LOW = {"A_G": 1, "A_R": 0}   # low FRET: more Cy3 light, so the higher A_G and the lower A_R
N_PE = frames["photoelectrons"].mean()
A_MEAN = {c[2:]: frames[c].mean() for c in ("A_B", "A_G", "A_R")}
assigned = seg[seg["state"] >= 0]
MEDIAN_FRAMES = {"low": assigned.loc[assigned["state"] != HIGH, "n_points"].median(),
                 "high": assigned.loc[assigned["state"] == HIGH, "n_points"].median()}
print(f"{len(frames):,} frames of {len(analysed):,} puncta; "
      f"mean {N_PE:.0f} photoelectrons per frame")
for col, name in [("A_G", "pixel 2"), ("A_R", "pixel 3")]:
    p, lo = PX[col], LOW[col]
    print(f"{name}: low FRET {p['mean'][lo]:.3f} ± {p['sd'][lo]:.3f}, high FRET {p['mean'][1 - lo]:.3f} ± "
          f"{p['sd'][1 - lo]:.3f}; difference {abs(np.diff(p['mean'])[0]):.3f}; 2 × mean sd {2 * p['sd'].mean():.3f}")
print(f"median frames per segment: low FRET {MEDIAN_FRAMES['low']:.0f}, high FRET {MEDIAN_FRAMES['high']:.0f}")

In [ ]:
# S84: apparent FRET efficiency of the segments
gmm, HIGH_E = tf.two_state_gmm(seg["E"])   # as section 5
inside = seg["E"].between(0, 1)
xs = np.linspace(0, 1, 500)
fig, ax = plotter.one_column_plot(height=3.5, width=3.33)
ax.hist(seg.loc[inside, "E"], bins=51, range=(0, 1), density=True, color="gray", label=r"$E_{\rm App}$")
for k, colour, label in [(1 - HIGH_E, "#000080", "Low FRET"), (HIGH_E, "#ff6600", "High FRET")]:
    mu, sd = gmm.means_.ravel()[k], np.sqrt(gmm.covariances_.ravel()[k])
    ax.plot(xs, gmm.weights_[k] * np.exp(-0.5 * ((xs - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)),
            color=colour, lw=1, label=label)
    ax.axvline(mu, color=colour, ls="--", lw=2)
    print(f"{label}: E_App = {mu:.2f} ± {sd:.2f} ({gmm.weights_[k]:.0%} of segments)")
ax.set(xlim=(0, 1), xlabel=r"$E_{\rm App}$", ylabel="probability density")
ax.legend(fontsize=6)
savefig(fig, "FRET_Apparent_Corrected_QY_fit_change.svg")
plt.show()

In [ ]:
# S85: resolvable shift against detected photoelectrons (Figure 2d bead fit, Ximea)
fractions = IOFunctions.IO_Functions(camera="zwo").pixel_fractions
N = np.linspace(50, 10000, 500)
fig, ax = plotter.one_column_plot(height=2.38, width=2.58)
for col, pix, name, colour in [("A_G", "G", "pixel 2", "darkgreen"), ("A_R", "R", "pixel 3", "darkred")]:
    ax.plot(N, tf.bead_resolution(N, A_MEAN, fractions, pix), color=colour, lw=1.5,
            label=f"{name}, precision-based")
    measured = 2 * PX[col]["sd"].mean()
    ax.scatter(N_PE, measured, s=25, color=colour, edgecolor="black", lw=0.5, alpha=0.6, zorder=3,
               label=f"{name}, experiment")
    print(f"{name}: experiment {measured:.3f}; bead fit at {N_PE:.0f} photoelectrons "
          f"{float(tf.bead_resolution(N_PE, A_MEAN, fractions, pix)):.3f}")
ax.set(xlim=(0, 10000), ylim=(0, 0.22), xlabel=r"N$_{\mathrm{photoelectrons}}$",
       ylabel="resolution/relative pixel QE")
ax.legend(fontsize=5)
savefig(fig, "EstimatedSpectral_Shift_ResolutionSI.svg")
plt.show()

In [ ]:
# S86: per-frame relative pixel QEs, two Gaussians each
fig, axs = plotter.two_column_plot(nrows=1, ncols=2, width=6.59, height=2.92)
for ax, col, name, colour in [(axs[0], "A_G", "pixel 2", "darkgreen"), (axs[1], "A_R", "pixel 3", "darkred")]:
    p, lo = PX[col], LOW[col]
    ax.hist(frames[col], bins=51, range=(0, 1), density=True, color=colour, alpha=0.5, label=f"{name} QE")
    for k, state, c in [(lo, "Low FRET", "#000080"), (1 - lo, "High FRET", "darkorange")]:
        pdf = p["weight"][k] * np.exp(-0.5 * ((xs - p["mean"][k]) / p["sd"][k]) ** 2) / (p["sd"][k] * np.sqrt(2 * np.pi))
        ax.plot(xs, pdf, color=c, lw=1, label=f"{state} = {p['mean'][k]:.2f}")
        ax.axvline(p["mean"][k], color=c, ls="--", lw=2)
    ax.set(xlim=(0, 1), xlabel=f"{name} QE")
    ax.legend(fontsize=6)
axs[0].set_ylabel("probability density")
savefig(fig, "pixel_QE_per_Frame_FRET.svg")
plt.show()

In [ ]:
# S87: resolvable change of relative pixel QE against frames per state
NF = np.arange(1, 101)
fig, ax = plotter.one_column_plot(height=3.41, width=3.26)
n_min = 1
for col, name, colour in [("A_G", "pixel 2", "darkgreen"), ("A_R", "pixel 3", "darkred")]:
    p, lo = PX[col], LOW[col]
    curve = tf.delta_resolution(NF, p["sd"][lo], p["sd"][1 - lo])
    shift = abs(np.diff(p["mean"])[0])
    ax.plot(NF, curve, color=colour, lw=2, label=f"{name} QE FRET resolution")
    ax.axhline(shift, color=colour, ls="--", lw=1)
    n_min = max(n_min, int(NF[curve <= shift].min()))
    print(f"{name}: observed change {shift:.3f}; resolvable at the median frames per segment: "
          f"{float(tf.delta_resolution(MEDIAN_FRAMES['low'], p['sd'][lo], p['sd'][1 - lo])):.3f} (low), "
          f"{float(tf.delta_resolution(MEDIAN_FRAMES['high'], p['sd'][lo], p['sd'][1 - lo])):.3f} (high)")
print(f"smallest N resolving both: {n_min} frames")
ax.axvline(n_min, color="black", lw=1.5)
ax.axvline(MEDIAN_FRAMES["low"], color="#000080", lw=1)
ax.axvline(MEDIAN_FRAMES["high"], color="darkorange", lw=1)
ax.set_xscale("log")
ax.set(xlabel=r"$N_{\rm frames}$", ylabel=r"$\Delta$ relative pixel QE")
ax.legend(loc="lower left", fontsize=6)
savefig(fig, "deltaPixel_vs_Frames.svg")
plt.show()

In [ ]:
# S88: six example fits of Pos-0-0, with the Bayer key of the sensor pixels under each crop
S88_SEED = 8
S88_QUERY = "chi_sqr < 3.5 and 12 < background_photons < 25 and 500 < photons < 1300"   # the paper's cut
tif0 = cp_files[0][0]
meta = sorted(FRET_ROOT.glob(tif0.name.split(".")[0] + "_metadata.txt"))[0]
roi_x, roi_y, _, _ = IO.metadata_reader_imageJ(str(meta))
fits0 = pd.read_hdf(tif0.with_name(tif0.name.split(".")[0] + ".h5"))
fits0 = fits0[fits0["puncta_id"].astype(str).radd("0_").isin(analysed)]   # puncta of the dwell analysis
examples_s88 = tf.example_fret_puncta(SupRes_Z, tif0, fits0, cal_zwo, (int(roi_x), int(roi_y)), roi_size=12,
                                      n_puncta=6, seed=S88_SEED, query=S88_QUERY, smoothing=smoothing())
for p in examples_s88:
    print(f"punctum {p['puncta_id']}, frame {p['frame']}, sensor origin (row, col) {p['origin']}, "
          f"fingerprint (B, G, R) {np.round(p['spectrum'], 2)}")
full_masks = SupRes_Z.mask.get_masks(*cal_zwo["gain_map"].shape)
fig, axs = tf.plot_example_fret_puncta(plotter, examples_s88, full_masks, FRET_PIXEL_NM)
savefig(fig, "example_puncta_raw_weights_fit_residual_spectrum_FRET_Pos-0-0.svg")
plt.show()